# TabPFN drift - validate the rebuilt engine (sudden stream, seed 0)
Goal: re-run the rebuilt policies with **real TabPFN** and compare with the teammate's
saved CSVs in `results/sudden_0_*.csv`. vanilla and sliding_window should match closely
(~1.0 before block 30, ~6.5 at block 30). Small differences are expected (random sampling,
TabPFN version). Report differences honestly; do not tune to force a match.

**Before you start:** Runtime > Change runtime type > **T4 GPU**. Run cells top to bottom.
If Colab disconnects, run again from cell 2: finished CSVs are skipped.

## 1. Check the GPU

In [ ]:
!nvidia-smi -L

## 2. Upload the code zip
Make it on your laptop, in PowerShell, inside the repo folder (this skips `.venv`):
```
tar.exe -a -cf ..\tabpfn-drift.zip common data eval methods results notebooks run_experiments.py smoke_test.py time_tabpfn.py plot_check.py requirements.txt
```
Then pick `tabpfn-drift.zip` (one folder up from the repo) when the upload box appears.

In [ ]:
from google.colab import files
files.upload()
!rm -rf /content/tabpfn-drift && mkdir -p /content/tabpfn-drift
!unzip -o -q tabpfn-drift.zip -d /content/tabpfn-drift
%cd /content/tabpfn-drift
!ls

## 3. Install packages (numpy, pandas, sklearn, torch are already on Colab)

In [ ]:
!pip install -q tabpfn river

## 4. TabPFN token (never paste it into a code cell or commit it)
Route A: licence key from https://ux.priorlabs.ai (accept the licence) -> `TABPFN_TOKEN`.
Route B: accept the terms of the TabPFN model on huggingface.co, then make a *Read* token -> `HF_TOKEN`.
Press Enter to skip either one.

In [ ]:
import os, getpass
t = getpass.getpass('TABPFN_TOKEN (Enter to skip): ').strip()
if t: os.environ['TABPFN_TOKEN'] = t
h = getpass.getpass('HF_TOKEN (Enter to skip): ').strip()
if h: os.environ['HF_TOKEN'] = h

## 5. Save outputs to Google Drive (survives disconnects)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/tabpfn-validation'
os.makedirs(OUT, exist_ok=True)
print('new results go to', OUT)

## 6. Pipeline self-test (stand-in model, not TabPFN). Must end with ALL SMOKE TESTS PASSED

In [ ]:
%cd /content/tabpfn-drift
!python smoke_test.py 2>&1 | tail -5

## 7. Time TabPFN (also proves the token works). Note the seconds for 2000 rows.

In [ ]:
!python time_tabpfn.py

## 8. Real TabPFN: vanilla + sliding_window on sudden, seed 0 (the main check)

In [ ]:
!python run_experiments.py --streams sudden --seeds 0 --methods vanilla sliding_window --c-max 2000 --out {OUT}

## 9. Compare with the teammate's results (solid = new, dashed = teammate)

In [ ]:
!python plot_check.py --results {OUT} --stream sudden --seed 0 --methods vanilla sliding_window --reference results --out {OUT}/validate_sudden0.png
from IPython.display import Image
Image(f'{OUT}/validate_sudden0.png')

**How to read it:** `pre(25-29)` should be ~1.0 and `block30` ~6.5 for both. `mean|diff|` is the average
per-block gap to the teammate's run. Write the numbers down as they are, even if they differ.

## 10. (Optional, slower) the other 4 policies on sudden seed 0. validation is ~5x slower.

In [ ]:
!python run_experiments.py --streams sudden --seeds 0 --methods recency adwin validation knn --c-max 2000 --out {OUT}
!python plot_check.py --results {OUT} --stream sudden --seed 0 --reference results --out {OUT}/validate_sudden0_all.png
Image(f'{OUT}/validate_sudden0_all.png')

Note: ADWIN's triangle appears one block later in the new run (e.g. 31 instead of 30). The detection
happens when block 30's labels arrive; our runner records the flag on the next predicted block.
The context is cut at the same moment either way.